# 1. Загрузка и черновая предобработка сырых таблиц

На этом шаге подготавливается единая рабочая версия сырых таблиц.  
Цель этапа — привести даты и идентификаторы к корректным типам, удалить заведомо лишние поля и выполнить минимальную таблиц-специфичную предобработку.

In [1]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import gc

from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path("C:\\Repos\\Xakaton\\data\\raw")

pd.set_option('display.max_rows', None)      # все строки
pd.set_option('display.max_columns', None)   # все столбцы
pd.set_option('display.width', None)         # не разрывать строки по ширине консоли
pd.set_option('display.max_colwidth', None)  # не обрезать текст внутри ячеек
pd.set_option('display.expand_frame_repr', False) # не переносить широкие таблицы на новые строки

## 1.1. Конфигурация загрузки

Ниже задаются:
- список таблиц, которые участвуют в основном пайплайне;
- колонки с датами;
- колонки с идентификаторами;
- поля, которые можно удалить сразу по описанию датасета.

In [3]:
# award_badges даже не читаем - она не содержит никакой полезной инфы (кроме того, что награда №1 - для олимпиадников)

FILES = {
    "users": "users.csv",
    "users_courses": "users_courses.csv",

    "groups": "groups.csv",

    "homework_items": "homework_items.csv",
    "homeworks": "homeworks.csv",

    "stats__module_1": "stats__module_1.csv",
    "stats__module_2": "stats__module_2.csv",
    "stats__module_3": "stats__module_3.csv",
    "stats__module_4": "stats__module_4.csv",

    "user_access_histories": "user_access_histories.csv",
    "user_activity_histories": "user_activity_histories.csv",
    "user_answers": "user_answers.csv",  # пока нет полного CSV
    "user_award_badges": "user_award_badges.csv",

    "lesson_tasks": "lesson_tasks.csv",
    "lessons": "lessons.csv",
    "user_lessons": "user_lessons.csv",

    "trainings": "trainings.csv",
    "user_trainings": "user_trainings.csv",

    "wk_users_courses_actions": "wk_users_courses_actions.csv",  # пока нет полного CSV
    "wk_media_view_sessions": "wk_media_view_sessions.csv",
}

DATE_COLS = {
    "users": [
        "created_at",
        "updated_at",
        "remember_created_at",
        "current_sign_in_at",
        "last_sign_in_at",
        "grade_changed_at",
        "d_updated_at",
    ],

    "users_courses": [
        "created_at",
        "updated_at",
        "access_finished_at",
        "wk_officially_started_at",
        "wk_course_completed_at",
    ],

    "groups": [
        "starts_at",
        "pupils_notified_at",
        "wk_actual_started_at",
        "wk_actual_finished_at",
    ],

    "homework_items": [
        # явных datetime-колонок нет
    ],

    "homeworks": [
        # явных datetime-колонок нет
    ],

    "stats__module_1": [
        "Дата зачисления",
        "Дата сдачи ПА (МСК)",
    ],

    "stats__module_2": [
        "Дата сдачи ПА (МСК)",
    ],

    "stats__module_3": [
        "Дата сдачи ПА (МСК)",
    ],

    "stats__module_4": [
        # явных datetime-колонок нет
    ],

    "user_access_histories": [
        "access_started_at",
        "access_expired_at",
    ],

    "user_activity_histories": [
        "created_at",
    ],

    "user_answers": [
        "submitted_at",
    ],

    "user_award_badges": [
        "created_at",
    ],

    "lesson_tasks": [
        # явных datetime-колонок нет
    ],

    "lessons": [
        "wk_attendance_tracking_disabled_at",
    ],

    "user_lessons": [
        # явных datetime-колонок нет
    ],

    "trainings": [
        "published_at",
    ],

    "user_trainings": [
        "started_at",
        "finished_at",
        "mark_saved_at",
    ],

    "wk_users_courses_actions": [
        "created_at",
        "updated_at",
    ],

    "wk_media_view_sessions": [
        "started_at",
    ],
}

## 1.2. Загрузка таблиц и приведение дат к корректному типу

In [4]:
def load_all_tables(data_dir: Path, files: dict, date_cols: dict):
    dfs = {}
    overview = []
    for name, filename in files.items():
        path = data_dir / filename
        if not path.exists():
            print(f'!!! Не найден файл: {path}')
            continue
        parse_dates = date_cols.get(name, [])
        df = pd.read_csv(path, index_col=0, parse_dates=parse_dates, low_memory=False)
        dfs[name] = df
        overview.append({
            'table': name,
            'rows': len(df),
            'cols': df.shape[1],
            'memory_mb': round(df.memory_usage(deep=True).sum() / (1024**2), 2),
            'parsed_dates': len(parse_dates)
        })
    overview_df = pd.DataFrame(overview).sort_values('rows', ascending=False).reset_index(drop=True)
    return dfs, overview_df

dfs, tables_overview = load_all_tables(DATA_DIR, FILES, DATE_COLS)
tables_overview

,table,rows,cols,memory_mb,parsed_dates
0,user_answers,15176182,14,7512.55,1
1,wk_users_courses_actions,12909207,7,3203.60,2
2,user_lessons,3070664,11,844.95,0
3,user_activity_histories,3031137,3,433.17,1
4,wk_media_view_sessions,852358,7,233.14,1
5,user_access_histories,667124,4,87.80,2
6,user_trainings,427628,12,141.96,3
7,users_courses,290835,13,105.55,5
8,user_award_badges,252843,3,21.22,1
9,users,95395,21,70.32,7


## 1.3. Удаление заведомо лишних полей

Часть колонок по официальному описанию датасета не несёт полезной информации для задачи или помечена как ненужная.  
Такие поля удаляются сразу, чтобы не тянуть технический шум дальше по пайплайну.

В частности:
- `user_answers.performance` помечено как ненужное;
- `wk_users_courses_actions.sourceable_id` помечено как ненужное;
- в `users` несколько технических полей также помечены как ненужные;
- в `award_badges` поле `name` и ссылка на картинку не нужны для анализа. :contentReference[oaicite:3]{index=3} :contentReference[oaicite:4]{index=4}